# 📊 Notebook 02 — Retrieval Evaluation
**Amlgo Labs RAG Chatbot Assignment**

This notebook evaluates the quality of the RAG pipeline using:
- **Hit@K** — Was the correct chunk in the top-K results?
- **MRR** — Mean Reciprocal Rank
- **End-to-End Q&A** — 5 success + 2 failure/edge cases

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.retriever import retrieve
from src.pipeline import stream_answer

sns.set_theme(style='darkgrid')
print('✅ Ready')

## Test Query Set

In [ ]:
# Define test queries with expected keywords in correct chunks
TEST_QUERIES = [
    {'query': 'What personal data is collected from users?',      'expected_keywords': ['collect', 'personal', 'data']},
    {'query': 'How long is user data retained?',                   'expected_keywords': ['retain', 'period', 'delete', 'store']},
    {'query': 'Can I request deletion of my account or data?',    'expected_keywords': ['delete', 'request', 'right', 'removal']},
    {'query': 'What third parties receive user information?',      'expected_keywords': ['third', 'party', 'share', 'disclose']},
    {'query': 'What cookies are used and for what purpose?',       'expected_keywords': ['cookie', 'tracking', 'browser']},
    {'query': 'How do I contact the company for privacy concerns?','expected_keywords': ['contact', 'email', 'privacy']},
    {'query': 'What are my rights under GDPR?',                    'expected_keywords': ['gdpr', 'right', 'subject', 'access']},
]

print(f'Evaluating {len(TEST_QUERIES)} queries...')

## Retrieval Metrics: Hit@K & MRR

In [ ]:
def keyword_hit(chunk_text, keywords):
    """Check if any keyword appears in chunk text (case-insensitive)."""
    text_lower = chunk_text.lower()
    return any(kw.lower() in text_lower for kw in keywords)

def evaluate_retrieval(queries, k_values=[1, 3, 5]):
    rows = []
    for item in queries:
        q = item['query']
        kws = item['expected_keywords']
        results = retrieve(q, k=max(k_values))

        hit_at = {}
        rr = 0.0
        for k in k_values:
            top_k = results[:k]
            hit_at[f'Hit@{k}'] = int(any(keyword_hit(r['text'], kws) for r in top_k))

        # MRR: reciprocal rank of first relevant result
        for rank, r in enumerate(results, start=1):
            if keyword_hit(r['text'], kws):
                rr = 1.0 / rank
                break

        top_score = results[0]['score'] if results else 0
        rows.append({'Query': q[:55] + '...', 'MRR': round(rr, 3),
                     'Top Score': round(top_score, 3), **hit_at})

    return pd.DataFrame(rows)

df_eval = evaluate_retrieval(TEST_QUERIES)
print('Results:')
print(df_eval.to_string(index=False))
print(f'\nMean MRR:   {df_eval["MRR"].mean():.3f}')
print(f'Mean Hit@1: {df_eval["Hit@1"].mean():.3f}')
print(f'Mean Hit@3: {df_eval["Hit@3"].mean():.3f}')
print(f'Mean Hit@5: {df_eval["Hit@5"].mean():.3f}')

In [ ]:
# Plot metric summary
metrics = ['Hit@1', 'Hit@3', 'Hit@5', 'MRR']
values  = [df_eval[m].mean() for m in metrics]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(metrics, values, color=['#3b82f6','#6366f1','#8b5cf6','#10b981'], width=0.5, edgecolor='white')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, 1.15)
ax.set_title('Retrieval Evaluation Metrics', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
plt.tight_layout()
plt.savefig('../chunks/retrieval_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## End-to-End Q&A Examples

In [ ]:
DEMO_QUERIES = [
    'What personal data is collected from users?',
    'How long is user data retained?',
    'Can I request deletion of my data?',
    'What are the user rights under this policy?',
    'What is something completely unrelated like quantum physics?',  # expected failure/edge case
]

for i, query in enumerate(DEMO_QUERIES, 1):
    print(f'\n{'='*70}')
    print(f'[Q{i}] {query}')
    print('─'*70)
    stream, sources = stream_answer(query)
    answer = ''
    for token in stream:
        answer += token
    print(f'Answer: {answer}')
    print(f'Sources: {len(sources)} chunks | Top score: {sources[0]["score"]:.3f if sources else "N/A"}')
